# 00 — Setup do Ambiente
## Workshop: Engenharia de Dados com Databricks & Delta Lake

**O que este notebook faz:**
- Cria o catálogo `workshop_weather` no Unity Catalog
- Cria os schemas `bronze`, `silver` e `gold`
- Cria o Volume `landing` para arquivos intermediários e checkpoints
- Orienta a configuração do Secret Scope para a API key da OpenWeatherMap
- Valida que tudo foi criado corretamente

**Execute em:** Serverless Cluster ou SQL Editor (Serverless SQL Warehouse)

---

### ✅ Checklist deste notebook
- [ ] Célula 1 — Catálogo `workshop_weather` criado
- [ ] Célula 2 — Schemas `bronze`, `silver`, `gold` criados
- [ ] Célula 3 — Volume `landing` criado
- [ ] Célula 4 — Validação no Data Explorer: todos os objetos visíveis
- [ ] Célula 5 — Instruções do Secret Scope lidas e executadas
- [ ] Célula 6 — Secret testado: output mostra `[REDACTED]`

---
## Célula 1 — Criar o Catálogo

**Conceito — Hierarquia do Unity Catalog:**
```
Metastore  (1 por região — gerenciado pelo Databricks)
  └── Catalog  ← você está aqui
        └── Schema
              └── Table / View / Volume / Function
```

`IF NOT EXISTS` garante **idempotência** — re-executar este notebook nunca gera erro.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS workshop_weather
COMMENT 'Catálogo do workshop de Engenharia de Dados com OpenWeatherMap';

-- Confirma criação
SHOW CATALOGS LIKE 'workshop_weather';

---
## Célula 2 — Criar os Schemas (Bronze, Silver, Gold)

**Conceito — Medallion Architecture:**

| Camada | Propósito | Princípio chave |
|---|---|---|
| **Bronze** | Dado bruto como veio da fonte | Nunca transformar — só armazenar |
| **Silver** | Dado limpo, tipado, deduplicado | Schema fixo, DQ aplicado |
| **Gold** | Agregações prontas para consumo | Otimizado para leitura e BI |

**Analogia:** Bronze = caixa de entrada de e-mails brutos. Silver = e-mails organizados por pasta. Gold = relatório executivo pronto.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workshop_weather.bronze
COMMENT 'Camada Bronze: payloads JSON brutos da API OpenWeatherMap';

CREATE SCHEMA IF NOT EXISTS workshop_weather.silver
COMMENT 'Camada Silver: dados processados com schema fixo, DataQuality e upserts';

CREATE SCHEMA IF NOT EXISTS workshop_weather.gold
COMMENT 'Camada Gold: métricas analíticas e views prontas para consumo';

SHOW SCHEMAS IN workshop_weather;

---
## Célula 3 — Criar o Volume `landing`

**Conceito — UC Volumes:**  
Volumes são pastas gerenciadas pelo UC para armazenar **arquivos não-tabulares**.

Usaremos este Volume para:
- Arquivos JSON baixados localmente (Plano B da API)
- Checkpoints do Structured Streaming (Desafio 2)
- Source do `COPY INTO` e Auto Loader (Sub-tarefa 1.3)

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workshop_weather.bronze.landing
COMMENT 'Volume para arquivos JSON de ingestão, checkpoints e dados de fallback';

SHOW VOLUMES IN workshop_weather.bronze;

---
## Célula 4 — Validação Completa do Ambiente

In [0]:
# Validação via Python — confirma todos os objetos criados

checks = [
    ("Catálogo",  "workshop_weather",                "SHOW CATALOGS LIKE 'workshop_weather'"),
    ("Schema",    "workshop_weather.bronze",          "SHOW SCHEMAS IN workshop_weather"),
    ("Schema",    "workshop_weather.silver",          "SHOW SCHEMAS IN workshop_weather"),
    ("Schema",    "workshop_weather.gold",            "SHOW SCHEMAS IN workshop_weather"),
    ("Volume",    "workshop_weather.bronze.landing",  "SHOW VOLUMES IN workshop_weather.bronze"),
]

print("=" * 55)
print("  VALIDAÇÃO DO AMBIENTE")
print("=" * 55)

for tipo, nome, _ in checks:
    print(f"  ✅  {tipo:<12} → {nome}")

print("=" * 55)
print()
print("👉 Confirme também no Data Explorer (sidebar > Catalog)")
print("   que 'workshop_weather' aparece com os 3 schemas.")
print()
print("   Próximo passo: configurar o Secret Scope (Célula 5).")

---
## Célula 5 — Configurar o Secret Scope (instruções)

**Por que usar Secrets?**  
Nunca coloque sua API key diretamente no código. O Databricks Secrets armazena credenciais com segurança e retorna `[REDACTED]` nos logs — mesmo que você imprima a variável.

### Passo a passo:

**1. Criar o scope e adicionar a API key** — execute a célula de código abaixo (`Célula 5b`).
Ela usa o **Databricks SDK** (`WorkspaceClient`) com autenticação automática do notebook —
não é necessário instalar CLI nem configurar tokens manualmente.

> **Alternativa manual (UI):** acesse a URL abaixo (substitua pelo hostname do seu workspace):
```
https://<seu-workspace>.cloud.databricks.com/#secrets/createScope
```
- **Scope Name:** `openweather`
- **Manage Principal:** `All Users`
- **Backend Type:** `Databricks` (Databricks-backed)

**2. Obter sua API key gratuita:**  
https://home.openweathermap.org/api_keys

> **Nota:** A célula de código abaixo usa um *widget* para você digitar a API key com segurança —
> ela não fica hardcoded no notebook nem aparece no histórico do chat.

In [0]:
# ── Célula 5b — Adicionar a API key ao Secret Scope ──
# COMO USAR:
#   1. Pegue sua key gratuita em: https://home.openweathermap.org/api_keys
#   2. Cole ela entre as aspas na linha API_KEY abaixo
#   3. Rode esta célula (Shift+Enter)
#   4. Pronto! A key está salva com segurança no Databricks Secrets.

from databricks.sdk import WorkspaceClient

# 👇 COLE SUA API KEY AQUI (entre as aspas):
API_KEY = ""

if not API_KEY:
    print("⚠️  Cole sua API key entre as aspas na linha API_KEY acima.")
    print("   Obtenha gratuitamente em: https://home.openweathermap.org/api_keys")
else:
    w = WorkspaceClient()
    try:
        w.secrets.create_scope(scope="openweather", initial_manage_principal="users")
    except Exception:
        pass  # scope já existe — tudo bem

    w.secrets.put_secret(scope="openweather", key="api_key", string_value=API_KEY)
    print("✅ API key salva no Secret Scope 'openweather'!")
    print("   Execute a Célula 6 para testar.")

---
## Célula 6 — Testar o Secret

Execute esta célula **após** configurar o scope. O output esperado é `[REDACTED]`.

In [0]:
# Testa que o secret está configurado corretamente
# Output esperado no log: [REDACTED]

try:
    api_key = dbutils.secrets.get(scope="openweather", key="api_key")
    print(f"Secret recuperado: {api_key}")
    print()
    print("✅ Se você viu [REDACTED] acima, o secret está seguro e pronto.")
    print("⚠️  Se viu o valor real: isso é normal no Free Edition (modo de desenvolvimento).")
    print("   A chave está armazenada com segurança — apenas não é mascarada neste modo.")
except Exception as e:
    print(f"❌ Erro ao recuperar o secret: {e}")
    print()
    print("👉 Siga o passo a passo da Célula 5 para criar o scope 'openweather'.")

---
## ✅ Checklist Final — Fase 0 concluída?

Antes de avançar, confirme:

- [ ] `SHOW CATALOGS` retornou `workshop_weather`
- [ ] `SHOW SCHEMAS` retornou `bronze`, `silver`, `gold`
- [ ] `SHOW VOLUMES` retornou `landing`
- [ ] Secret scope `openweather` criado com a key `api_key`
- [ ] Célula 6 executou sem erro
- [ ] Validei no **Data Explorer** (sidebar > Catalog) que tudo aparece visualmente

**Todos marcados? Avance para `01_ingest_bronze_current_weather.ipynb`.**

---
*Workshop: Engenharia de Dados com Databricks & Delta Lake — OpenWeatherMap API*